# AIVLE스쿨 2차 미니프로젝트: 신규 아파트 주차 수요 예측

<img src = "https://github.com/Jangrae/img/blob/master/parking.png?raw=true" width=800, align="left"/>

# 단계 1: 데이터 전처리

## [미션]

단지별 등록 차량 수를 예측하기에 적합한 형태로 데이터 전처리를 수행합니다.

1) 필요한 변수를 추가하고 불필요한 변수를 제거합니다.
2) 단지별 데이터와 상세 데이터를 분리합니다.
3) 상세 데이터를 단지별로 집계합니다.
    - 단지별 총면적 집계
    - 전용면적구간 집계 (피벗형태)
    - 단지별 임대보증금, 임대료 평균 집계
4) 단지별 데이터와 집계 데이터를 하나로 합칩니다.
5) 변수 추가 (옵션)
    - 등록 차량수를 예측하기 위해 필요한 변수를 추가합니다.

## 1. 환경설정

### (1) 로컬 수행(Anaconda)

- project 폴더에 필요한 파일들을 넣고, 본 파일을 열었다면, 별도 경로 지정이 필요하지 않습니다.

In [2]:
# 기본 경로
path = ''

### (4) 라이브러리 불러오기

In [5]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import joblib
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'

### (5) 데이터 불러오기

- 학습용 데이터(train.xlsx)를 읽어옵니다.
- 읽어온 데이터를 apart 데이터프레임으로 선언합니다.
- 평가용 데이터(test.xlsx) 파일은 모델 완성 후 사용할 미래의 데이터입니다.

#### 1) 데이터 불러오기

In [8]:
# 파일 불러오기
apart = pd.read_excel(path+'train.xlsx')

#### 2) 기본 정보 조회

In [10]:
apart

,단지코드,단지명,총세대수,전용면적별세대수,지역,준공일자,건물형태,난방방식,승강기설치여부,단지내주차면수,전용면적,공용면적,임대보증금,임대료,실차량수
0,C0001,엘에이치 서초4단지,78,35,서울,20131204,계단식,개별가스난방,전체동 설치,120,51.89,19.2603,50758000,620370,109
1,C0001,엘에이치 서초4단지,78,43,서울,20131204,계단식,개별가스난방,전체동 설치,120,59.93,22.2446,63166000,665490,109
2,C0002,LH삼성아파트,35,26,서울,20130801,복도식,개별가스난방,전체동 설치,47,27.75,16.5375,63062000,458640,35
3,C0002,LH삼성아파트,35,9,서울,20130801,복도식,개별가스난방,전체동 설치,47,29.08,17.3302,63062000,481560,35
4,C0003,강남LH8단지,88,7,서울,20131023,계단식,개별가스난방,전체동 설치,106,59.47,21.9462,72190000,586540,88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1152,C0356,덕유마을 주공1단지,956,956,경기,19941130,복도식,지역가스난방,전체동 설치,202,26.37,12.7500,9931000,134540,243
1153,C0358,화천신읍(공공실버) 영구임대,120,66,강원,20200110,복도식,NaN,전체동 설치,40,24.83,15.1557,2129000,42350,47
1154,C0358,화천신읍(공공실버) 영구임대,120,54,강원,20200110,복도식,NaN,전체동 설치,40,33.84,20.6553,2902000,57730,47
1155,C0359,영천야사4,447,149,대구경북,19940615,복도식,중앙유류난방,전체동 설치,89,26.37,13.3800,7134000,118880,78


In [14]:
# 표시 형식 변경
pd.set_option('display.float_format', '{:.4f}'.format)

In [16]:
apart.describe()

,총세대수,전용면적별세대수,준공일자,단지내주차면수,전용면적,공용면적,임대보증금,임대료,실차량수
count,1157.0000,1157.0000,1157.0000,1157.0000,1157.0000,1157.0000,1157.0000,1157.0000,1157.0000
mean,659.0752,163.6914,20086669.8099,682.2619,51.5656,20.5624,28507893.6906,225940.8816,650.7623
std,456.1106,166.7664,67779.8549,473.3318,18.2433,5.1644,28906872.0789,176810.2416,390.5735
min,1.0000,1.0000,19920101.0000,10.0000,17.5900,5.8500,0.0000,0.0000,21.0000
25%,315.0000,44.0000,20050310.0000,308.0000,39.4800,16.9974,13797000.0000,117740.0000,320.0000
50%,595.0000,112.0000,20100415.0000,629.0000,46.9000,20.3847,19973000.0000,184290.0000,626.0000
75%,918.0000,229.0000,20131212.0000,911.0000,59.8100,23.7225,33753000.0000,263440.0000,894.0000
max,2289.0000,1258.0000,20220710.0000,4553.0000,139.3500,42.7600,254922000.0000,1058030.0000,1657.0000


In [18]:
apart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1157 entries, 0 to 1156
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   단지코드      1157 non-null   object 
 1   단지명       1157 non-null   object 
 2   총세대수      1157 non-null   int64  
 3   전용면적별세대수  1157 non-null   int64  
 4   지역        1157 non-null   object 
 5   준공일자      1157 non-null   int64  
 6   건물형태      1135 non-null   object 
 7   난방방식      1082 non-null   object 
 8   승강기설치여부   1059 non-null   object 
 9   단지내주차면수   1157 non-null   int64  
 10  전용면적      1157 non-null   float64
 11  공용면적      1157 non-null   float64
 12  임대보증금     1157 non-null   int64  
 13  임대료       1157 non-null   int64  
 14  실차량수      1157 non-null   int64  
dtypes: float64(2), int64(7), object(6)
memory usage: 135.7+ KB


In [20]:
apart.corr(numeric_only=True)

,총세대수,전용면적별세대수,준공일자,단지내주차면수,전용면적,공용면적,임대보증금,임대료,실차량수
총세대수,1.0000,0.4865,0.0586,0.6157,-0.2062,-0.0161,0.0679,0.0790,0.6888
전용면적별세대수,0.4865,1.0000,-0.0527,0.2294,-0.1945,-0.1144,-0.0752,-0.0280,0.2941
준공일자,0.0586,-0.0527,1.0000,0.3071,0.2292,0.3192,0.3344,0.3635,0.3455
단지내주차면수,0.6157,0.2294,0.3071,1.0000,0.3024,0.3451,0.3051,0.1991,0.8906
전용면적,-0.2062,-0.1945,0.2292,0.3024,1.0000,0.7541,0.5539,0.4781,0.3143
공용면적,-0.0161,-0.1144,0.3192,0.3451,0.7541,1.0000,0.5167,0.4732,0.3498
임대보증금,0.0679,-0.0752,0.3344,0.3051,0.5539,0.5167,1.0000,0.8267,0.3500
임대료,0.0790,-0.0280,0.3635,0.1991,0.4781,0.4732,0.8267,1.0000,0.3254
실차량수,0.6888,0.2941,0.3455,0.8906,0.3143,0.3498,0.3500,0.3254,1.0000


## 2. 데이터 전처리 ①

- 결측치 존재 여부를 확인하고 적절히 처리합니다.
- 필요한 변수를 추가하고, 불필요한 변수를 제거합니다.

### (1) 결측치 처리

- 결측치가 있는 지 확인합니다.

In [24]:
apart.isna().sum()

단지코드         0
단지명          0
총세대수         0
전용면적별세대수     0
지역           0
준공일자         0
건물형태        22
난방방식        75
승강기설치여부     98
단지내주차면수      0
전용면적         0
공용면적         0
임대보증금        0
임대료          0
실차량수         0
dtype: int64

In [26]:
apart['건물형태'].value_counts()

건물형태
복도식    623
계단식    321
혼합식    191
Name: count, dtype: int64

In [28]:
apart['난방방식'].value_counts()

난방방식
개별가스난방    568
지역난방      333
지역가스난방    120
중앙가스난방     44
중앙난방       11
중앙유류난방      3
지역유류난방      2
개별유류난방      1
Name: count, dtype: int64

In [30]:
apart['승강기설치여부'].value_counts()

승강기설치여부
전체동 설치    1030
미설치         18
일부동 설치      11
Name: count, dtype: int64

- 결측치는 적절한 값으로 채웁니다.
- 예를 들어 범주형 변수인 경우는 각 변수의 최빈값으로 채울 수 있습니다.

In [33]:
apart['건물형태'].fillna(apart['건물형태'].mode()[0], inplace=True)
apart['난방방식'].fillna(apart['난방방식'].mode()[0], inplace=True)
apart['승강기설치여부'].fillna(apart['승강기설치여부'].mode()[0], inplace=True)

In [35]:
apart.isna().sum()

단지코드        0
단지명         0
총세대수        0
전용면적별세대수    0
지역          0
준공일자        0
건물형태        0
난방방식        0
승강기설치여부     0
단지내주차면수     0
전용면적        0
공용면적        0
임대보증금       0
임대료         0
실차량수        0
dtype: int64

### (2) 변수 추가

- '준공일자' 변수 값 앞 4 자리를 갖는 int 형 변수 '준공연도'를 추가합니다.
- 총면적 = (전용면적 + 공용면적) * 전용면적별세대수 공식에 따른'총면적' 변수를 추가합니다.

In [38]:
apart['준공연도'] = apart['준공일자']//10000

In [40]:
apart['총면적'] = (apart['전용면적'] + apart['공용면적']) * apart['전용면적별세대수']

In [42]:
apart.head()

,단지코드,단지명,총세대수,전용면적별세대수,지역,준공일자,건물형태,난방방식,승강기설치여부,단지내주차면수,전용면적,공용면적,임대보증금,임대료,실차량수,준공연도,총면적
0,C0001,엘에이치 서초4단지,78,35,서울,20131204,계단식,개별가스난방,전체동 설치,120,51.8900,19.2603,50758000,620370,109,2013,2490.2605
1,C0001,엘에이치 서초4단지,78,43,서울,20131204,계단식,개별가스난방,전체동 설치,120,59.9300,22.2446,63166000,665490,109,2013,3533.5078
2,C0002,LH삼성아파트,35,26,서울,20130801,복도식,개별가스난방,전체동 설치,47,27.7500,16.5375,63062000,458640,35,2013,1151.4750
3,C0002,LH삼성아파트,35,9,서울,20130801,복도식,개별가스난방,전체동 설치,47,29.0800,17.3302,63062000,481560,35,2013,417.6918
4,C0003,강남LH8단지,88,7,서울,20131023,계단식,개별가스난방,전체동 설치,106,59.4700,21.9462,72190000,586540,88,2013,569.9134


### (3) 불필요한 변수 제거

- '단지명' 변수는 단일값을 가지므로 제거합니다.
- '단지내주차면수' 변숫값을 기반으로 등록 차량수를 예측하는 것은 의미가 없으니, '단지내주차면수' 변수를 제거합니다.
- '준공연도' 변수를 추가했으니 '준공일자' 변수를 제거합니다.

In [45]:
drop_cols = ['단지명', '단지내주차면수', '준공일자']
apart.drop(columns=drop_cols, inplace=True)

In [47]:
apart.head()

,단지코드,총세대수,전용면적별세대수,지역,건물형태,난방방식,승강기설치여부,전용면적,공용면적,임대보증금,임대료,실차량수,준공연도,총면적
0,C0001,78,35,서울,계단식,개별가스난방,전체동 설치,51.8900,19.2603,50758000,620370,109,2013,2490.2605
1,C0001,78,43,서울,계단식,개별가스난방,전체동 설치,59.9300,22.2446,63166000,665490,109,2013,3533.5078
2,C0002,35,26,서울,복도식,개별가스난방,전체동 설치,27.7500,16.5375,63062000,458640,35,2013,1151.4750
3,C0002,35,9,서울,복도식,개별가스난방,전체동 설치,29.0800,17.3302,63062000,481560,35,2013,417.6918
4,C0003,88,7,서울,계단식,개별가스난방,전체동 설치,59.4700,21.9462,72190000,586540,88,2013,569.9134


## 3. 데이터 전처리 ②

- 단지별 데이터와 상세 데이터로 분리합니다.
- 상세 데이터를 3가지 형태로 집계합니다.
- 단지별 데이터와 상세 데이터 집계 결과를 조인(Merge) 합니다.

### (1) 데이터 분리

- 단지별 데이터를 갖는 data01 데이터프레임을 선언합니다.
- 상세 데이터를 갖는 data02 데이터프레임을 선언합니다.

#### 1) 단지별 데이터 분리

- 다음 열을 갖는 data01 데이터프레임으로 선언합니다.
    - '단지코드', '총세대수', '지역', '준공연도', '건물형태', '난방방식', '승강기설치여부', '실차량수'
- data01 데이터프레임의 중복행을 제거합니다.
- 인덱스를 초기화 합니다. (단, 기존 인덱스 제거)
- 중복행 제거 여부를 필히 확인합니다.

In [281]:
data01 = pd.DataFrame()
data01 = apart[['단지코드', '총세대수', '지역', '준공연도', '건물형태', '난방방식', '승강기설치여부', '실차량수']]

In [283]:
data01.drop_duplicates(inplace=True)

In [285]:
data01.duplicated().sum()

0

In [287]:
data01.reset_index(drop=True, inplace=True)

In [289]:
data01.head()

,단지코드,총세대수,지역,준공연도,건물형태,난방방식,승강기설치여부,실차량수
0,C0001,78,서울,2013,계단식,개별가스난방,전체동 설치,109
1,C0002,35,서울,2013,복도식,개별가스난방,전체동 설치,35
2,C0003,88,서울,2013,계단식,개별가스난방,전체동 설치,88
3,C0004,477,서울,2014,복도식,지역난방,전체동 설치,943
4,C0006,15,서울,2013,복도식,개별가스난방,전체동 설치,21


#### 2) 상세 데이터 분리
    
- 다음 열을 갖는 data02 데이터프레임으로 선언합니다.
    - '단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료'

In [292]:
data02 = pd.DataFrame()
data02 = apart[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]

In [294]:
data02.head()

,단지코드,총면적,전용면적별세대수,전용면적,공용면적,임대보증금,임대료
0,C0001,2490.2605,35,51.8900,19.2603,50758000,620370
1,C0001,3533.5078,43,59.9300,22.2446,63166000,665490
2,C0002,1151.4750,26,27.7500,16.5375,63062000,458640
3,C0002,417.6918,9,29.0800,17.3302,63062000,481560
4,C0003,569.9134,7,59.4700,21.9462,72190000,586540


### (2) 상세 데이터 집계

- 앞에서 선언한 data02 데이터프레임을 대상으로 다음 3가지 형태로 집계합니다.
    - 단지코드별 총면적 합을 집계합니다.
    - 전용면적을 의미있는 구간으로 나누어 피벗 형태로 집계합니다.
    - 단지코드별 임대보증금, 임대료 평균을 집계합니다

#### 1) 단지코드별 총면적 합 집계

- 단지코드별 총면적 합을 집계합니다.
- 집계한 결과를 df_area 데이터프레임으로 선언합니다.

In [298]:
df_area = data02.groupby('단지코드', as_index=False)[['총면적']].sum()

In [300]:
df_area.head()

,단지코드,총면적
0,C0001,6023.7683
1,C0002,1569.1668
2,C0003,7180.1396
3,C0004,47058.9273
4,C0006,543.0268


#### 2) 전용면적 구간별 집계 (피벗 형태)

- data02 데이터프레임에 전용면적을 몇몇 구간으로 나눈 범줏값을 갖는 변수를 추가합니다.
- 구간을 어떻게 나눌 지 충분히 고민해 봅니다.
    - 구간 예: 10-30, 30-40, 40-50, 50-60, 60-70, 70-80, 80-200 
- 추가할 변수 이름은 '전용면적구간'으로 합니다.
- 참고: pd.cut() 함수를 활용합니다.

In [303]:
data02['전용면적구간'] = pd.cut(data02['전용면적'],
                             bins=[10, 40, 60, 80, 200], 
                             labels=['면적10_40', '면적40_60', '면적60_80', '면적80_200'])

In [305]:
data02.head()

,단지코드,총면적,전용면적별세대수,전용면적,공용면적,임대보증금,임대료,전용면적구간
0,C0001,2490.2605,35,51.8900,19.2603,50758000,620370,면적50_55
1,C0001,3533.5078,43,59.9300,22.2446,63166000,665490,면적55_60
2,C0002,1151.4750,26,27.7500,16.5375,63062000,458640,면적10_40
3,C0002,417.6918,9,29.0800,17.3302,63062000,481560,면적10_40
4,C0003,569.9134,7,59.4700,21.9462,72190000,586540,면적55_60


- 단지코드, 전용면적구간별 전용면적별세대수 합을 집계합니다.
- 집계 결과를 temp 데이터프레임으로 선언합니다.

In [308]:
temp = data02.groupby(['단지코드', '전용면적구간'], as_index=False)[['전용면적별세대수']].sum()

In [310]:
temp.head()

,단지코드,전용면적구간,전용면적별세대수
0,C0001,면적10_40,0
1,C0001,면적40_50,0
2,C0001,면적50_55,35
3,C0001,면적55_60,43
4,C0001,면적60_80,0


- temp 데이터프레임을 피벗 형태로 변환하여 df_pivot 데이터프레임으로 선언합니다.
- 인덱스를 초기화합니다. (단, 인덱스였던 '단지코드'가 제거되면 안됨)
- 이후 작업의 편의를 위해 일반적인 데이터프레임 형태를 갖게 합니다.
- 참고: df2 = df1.pivot(index=?, columns=?, values=?) 형태로 pivot() 메서드를 사용합니다.
- 참고: df2.columns.name=None 형태의 구문을 사용해 열이름에 대한 이름을 제거합니다.

In [313]:
df_pivot = temp.pivot(index='단지코드', columns='전용면적구간', values='전용면적별세대수')
df_pivot.reset_index(drop=False, inplace=True)
df_pivot.columns.name=None
df_pivot

,단지코드,면적10_40,면적40_50,면적50_55,면적55_60,면적60_80,면적80_200
0,C0001,0,0,35,43,0,0
1,C0002,35,0,0,0,0,0
2,C0003,0,0,0,88,0,0
3,C0004,0,0,0,150,216,111
4,C0006,15,0,0,0,0,0
...,...,...,...,...,...,...,...
340,C1341,140,0,0,0,0,0
341,C1354,1369,0,17,0,0,0
342,C2307,196,0,0,0,0,0
343,C2343,80,0,0,0,0,0


#### 3) 임대보증금, 임대료 평균 집계

- 단지코드별 임대보증금, 임대료 평균을 집계합니다.
- 집계 결과를 df_rent 데이터프레임으로 선언합니다.

In [316]:
df_rent = data02.groupby('단지코드', as_index=False)[['임대보증금', '임대료']].mean()
df_rent.head()

,단지코드,임대보증금,임대료
0,C0001,56962000.0000,642930.0000
1,C0002,63062000.0000,470100.0000
2,C0003,72190000.0000,586540.0000
3,C0004,101516666.6667,950305.0000
4,C0006,55227500.0000,340148.3333


In [318]:
df_rent_t=data02.groupby(['단지코드','전용면적별세대수'], as_index=False)[['임대보증금','임대료']].mean()
df_rent_t['임대보증금'] = df_rent_t['임대보증금'] * df_rent_t['전용면적별세대수']
df_rent_t['임대료'] = df_rent_t['임대료'] * df_rent_t['전용면적별세대수']
df_rent_t1 = df_rent_t.groupby(['단지코드'], as_index=False)[['임대보증금','임대료']].sum()
df_rent_t2 = df_rent_t.groupby(['단지코드'], as_index=False)[['전용면적별세대수']].sum()

df_rent_t = pd.merge(df_rent_t1, df_rent_t2, on='단지코드')
df_rent_t['임대보증금'] = df_rent_t['임대보증금'] / df_rent_t['전용면적별세대수']
df_rent_t['임대료'] = df_rent_t['임대료'] / df_rent_t['전용면적별세대수']
df_rent_t.drop(columns='전용면적별세대수', inplace=True)
df_rent_t.head()

,단지코드,임대보증금,임대료
0,C0001,57598307.6923,645243.8462
1,C0002,63062000.0000,464533.7143
2,C0003,72190000.0000,586540.0000
3,C0004,91584727.4633,877880.2725
4,C0006,54463150.0000,330212.0000


### (3) 집계 결과 합치기

- 위 과정에서 만든 df_area, df_pivot, df_rent 데이터프레임을 data01 데이터프레임과 조인(Merge)합니다.
- data01 데이터프레임이 기준 데이터프레임입니다.
- '단지코드' 변수가 조인 기준이 되며, how='left'를 지정합니다.
- 조인 결과를 base_data 데이터프레임으로 선언합니다.

In [323]:
base_data.loc[base_data['단지코드'] == 'C1354']

,단지코드,총세대수,지역,준공연도,건물형태,난방방식,승강기설치여부,실차량수,총면적,면적10_40,면적40_50,면적50_55,면적55_60,면적60_80,면적80_200,임대보증금,임대료
341,C1354,1386,대전충남,1993,복도식,중앙가스난방,전체동 설치,258,57616.8100,1369,0,17,0,0,0,6079848.0176,84027.5184


## 4. 데이터 셋 저장

- joblib.dump() 함수를 사용하여 최종 데이터 셋을 파일로 저장합니다.
- 파일 이름은 base_data1.pkl로 합니다.

In [326]:
# 파일로 저장
joblib.dump(base_data, path+'base_data3.pkl')

['base_data3.pkl']

In [ ]:
가격대를 범주형으로 바꿔서 분석하는 방법!